In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn import preprocessing

# Train DCNN

In [2]:
class MelSegmentDataset(Dataset):
    """Dataset for loading mel-spectrogram segments and their corresponding labels."""
    def __init__(self, x_path, y_path):
        self.X = np.load(x_path)
        self.y = np.load(y_path)
        
        # Add ImageNet normalization to your transforms
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((227, 227), antialias=True),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        # 1. Get the 3-channel matrix
        img = self.X[idx].copy() 
        
        # 2. Min-Max Scale each channel independently to [0, 1]
        for c in range(3):
            min_val = img[c].min()
            max_val = img[c].max()
            if max_val > min_val:
                img[c] = (img[c] - min_val) / (max_val - min_val)
            else:
                img[c] = 0.0 # Fallback for empty/silent frames
                
        # 3. Transpose from (C, H, W) to (H, W, C) for ToTensor()
        img = img.transpose(1, 2, 0)
        
        # 4. Apply resizing and ImageNet normalization
        return self.transform(img).float(), torch.tensor(self.y[idx], dtype=torch.long)
    
train_dataset       = MelSegmentDataset("processed_data_emodb/X_train.npy", "processed_data_emodb/y_train.npy")
validation_dataset  = MelSegmentDataset("processed_data_emodb/X_validation.npy", "processed_data_emodb/y_validation.npy")
test_dataset        = MelSegmentDataset("processed_data_emodb/X_test.npy", "processed_data_emodb/y_test.npy")

train_loader        = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader   = DataLoader(validation_dataset, batch_size=32, shuffle=False)
test_loader         = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [3]:
# Initialize pre-trained AlexNet [cite: 149]
model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Modify the final layer for 7 EMO-DB classes instead of 1000 [cite: 245]
num_ftrs = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_ftrs, 7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [4]:
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(dataloader)

def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    return running_loss / len(dataloader), accuracy

In [5]:
# Training setup (SGD, lr=0.001, momentum=0.9) [cite: 410]
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

best_val_loss = float('inf') # To track the best validation loss for early stopping
PATIENCE = 5  # Number of epochs to wait for improvement before stopping
counter = 0  # Counter for early stopping

# Training loop with 10 epochs for demonstration (set to 300 for full replication) [cite: 411]
for epoch in range(50): 
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_accuracy = evaluate(model, validation_loader, criterion)
    
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

    # early stopping condition (if validation loss increases) can be implemented here for better performance
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0  # reset counter if validation loss improves

        # Save the best model based on validation loss
        torch.save(model.state_dict(), "fine_tuned_alexnet_best.pth")
        print("Validation loss improved, model saved.")

    else:
        counter += 1
        print (f"Validation loss did not improve. Counter: {counter}/{PATIENCE}")
        if counter >= PATIENCE:
            print("Validation loss increased, stopping early.")
            break


Epoch 1, Train Loss: 1.5331, Val Loss: 1.3132, Val Accuracy: 0.4842
Validation loss improved, model saved.
Epoch 2, Train Loss: 1.2204, Val Loss: 1.0562, Val Accuracy: 0.5885
Validation loss improved, model saved.
Epoch 3, Train Loss: 1.0908, Val Loss: 1.0382, Val Accuracy: 0.6089
Validation loss improved, model saved.
Epoch 4, Train Loss: 0.9435, Val Loss: 0.9735, Val Accuracy: 0.5996
Validation loss improved, model saved.
Epoch 5, Train Loss: 0.8909, Val Loss: 1.1618, Val Accuracy: 0.5624
Validation loss did not improve. Counter: 1/5
Epoch 6, Train Loss: 0.7985, Val Loss: 0.9225, Val Accuracy: 0.6220
Validation loss improved, model saved.
Epoch 7, Train Loss: 0.6941, Val Loss: 0.8740, Val Accuracy: 0.6797
Validation loss improved, model saved.
Epoch 8, Train Loss: 0.6223, Val Loss: 0.9758, Val Accuracy: 0.6164
Validation loss did not improve. Counter: 1/5
Epoch 9, Train Loss: 0.5795, Val Loss: 1.1961, Val Accuracy: 0.6127
Validation loss did not improve. Counter: 2/5
Epoch 10, Train 

# DTPM

Checkpoint: Can Run DCNN and DTPM seperately, as the intermediate model trained from DCNN is saved in disk and reloaded again

In [6]:
from torchvision import models, transforms
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [7]:
# ==========================================
# 1. SETUP THE FEATURE EXTRACTOR (DCNN)
# ==========================================
print("Loading fine-tuned DCNN...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Rebuild the AlexNet structure matching Phase 2 
model = models.alexnet()
num_ftrs = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_ftrs, 7)

# Load your successful weights from the Phase 2 training loop
model.load_state_dict(torch.load("fine_tuned_alexnet_best.pth", map_location=device))

# Expose FC7: Keep layers 0 through 4 (Drop, Lin4096, ReLU, Drop, Lin4096) [cite: 247-248]
# FC7 is the second last layer of AlexNet, so we keep everything up to that point and discard the final classification layer.
model.classifier = nn.Sequential(*list(model.classifier.children())[:5])
model.eval()    # Lock the model weights
model = model.to(device)

# Identical image normalization pipeline from Phase 2
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((227, 227), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_fc7(segments_array):
    """
    Take the array of raw 64 * 64 * 3 segments belonging to one specific audio file.
    For each segment:
    1. Min-Max Scale each channel independently to [0, 1]
    2. Transpose from (C, H, W) to (H, W, C) for ToTensor()
    3. Apply resizing and ImageNet normalization
    4. Push the batch of processed segments through the DCNN and extract the FC7 features (4096-1) for each segment.

    Return a numpy array of shape (N_segments, 4096) containing the FC7 features for all segments of that audio file.
    """
    processed_tensors = []
    
    for img in segments_array:
        img_copy = img.copy()
        # Min-Max Scaling per channel   
        for c in range(3):
            min_v, max_v = img_copy[c].min(), img_copy[c].max()
            if max_v > min_v: img_copy[c] = (img_copy[c] - min_v) / (max_v - min_v)
            else: img_copy[c] = 0.0
                
        # Transpose to (H, W, C), apply PyTorch transforms, add batch dimension
        img_transposed = img_copy.transpose(1, 2, 0)
        tensor = transform(img_transposed).unsqueeze(0)
        processed_tensors.append(tensor)
        
    # Stack into a batch and push through network
    batch_tensor = torch.cat(processed_tensors).to(device)
    with torch.no_grad():
        return model(batch_tensor).cpu().numpy() # Returns shape: (N_segments, 4096)

Loading fine-tuned DCNN...


In [8]:
# ==========================================
# 2. DISCRIMINANT TEMPORAL PYRAMID MATCHING (DTPM)
# ==========================================
def lp_norm_pooling(matrix, p=1.12):
    # The optimized pooling formula [cite: 302, 801-802]
    return np.power(np.mean(np.power(np.abs(matrix), p), axis=0), 1/p)
# p = 1.12 is optimal for emo-db

def dtpm(segment_features, L=2):
    """Groups chronological vectors into L levels and pools them."""
    N = segment_features.shape[0] # Number of segments for this audio file
    final_feature = []
    
    # L=0: Whole utterance (Weight: 1/4) [cite: 299, 311]
    # In pyramid matching, the global view gets a lower weight because it lacks specific timeline details.
    final_feature.append(lp_norm_pooling(segment_features) * (1 / (2**L)))
    
    # L=1: Halves (Weight: 1/4) [cite: 300]
    if N >= 2:
        mid = N // 2
        final_feature.append(lp_norm_pooling(segment_features[:mid]) * (1 / (2**L)))
        final_feature.append(lp_norm_pooling(segment_features[mid:]) * (1 / (2**L)))
    else: # Fallback if audio is too short
        final_feature.extend([np.zeros(4096)] * 2)
        
    # L=2: Quarters (Weight: 1/2) [cite: 301]
    # The paper assigns the highest weight to the smallest chunks because they contain the most precise, localized emotional changes (like a sudden voice crack). This adds 4 new vectors to the list.
    if L == 2:
        if N >= 4:
            q_size = N // 4
            for i in range(4):
                start = i * q_size
                end = (i + 1) * q_size if i != 3 else N
                final_feature.append(lp_norm_pooling(segment_features[start:end]) * (1 / (2**(L-1))))
        else:
            final_feature.extend([np.zeros(4096)] * 4)

    # If number of segements is less than 4 (ie very short audio), we pad with zeros to maintain consistent feature length.
            
    # Combine into single 28,672-D vector
    # Concetaned from 7 arrays of 4096-D each (1 global + 2 halves + 4 quarters) [cite: 308-309]
    return np.concatenate(final_feature)

In [9]:
# ==========================================
# 3. DATA PROCESSING PIPELINE
# ==========================================
def process_dataset_to_global_features(split_name):
    print(f"\nProcessing {split_name} set...")
    X_segs = np.load(f"processed_data_emodb/X_{split_name}.npy")
    y_lbls = np.load(f"processed_data_emodb/y_{split_name}.npy")
    u_ids = np.load(f"processed_data_emodb/utterance_ids_{split_name}.npy")
    
    # Group segments back into their original audio files
    utterance_dict = {}
    for i, uid in enumerate(u_ids):
        if uid not in utterance_dict:
            utterance_dict[uid] = {'segments': [], 'label': y_lbls[i]}
        utterance_dict[uid]['segments'].append(X_segs[i])
        
    X_global, y_global = [], []
    
    # Process each utterance
    for uid, data in utterance_dict.items():
        # 1. Extract FC7 features for all segments in this utterance
        fc7_feats = extract_fc7(np.array(data['segments']))
        # 2. Pool them temporally using DTPM
        global_feat = dtpm(fc7_feats, L=2)
        
        X_global.append(global_feat)
        y_global.append(data['label'])
        
    return np.array(X_global), np.array(y_global)

# Run the pipeline on your previously separated datasets
X_train_global, y_train_global = process_dataset_to_global_features("train")
# (Optional) You can process validation too if you want to tune SVM hyperparams
# X_val_global, y_val_global = process_dataset_to_global_features("validation")
X_test_global, y_test_global = process_dataset_to_global_features("test")


Processing train set...

Processing test set...


In [10]:
# ==========================================
# 4. SVM CLASSIFICATION
# ==========================================
print("\nTraining Linear SVM on DTPM features...")
# Using Linear Kernel and One-vs-One strategy [cite: 413, 426]
svm_classifier = SVC(kernel='linear', decision_function_shape='ovo')
svm_classifier.fit(X_train_global, y_train_global)

print("Evaluating on unseen Test Set (Speakers 15 & 16)...")
predictions = svm_classifier.predict(X_test_global)

accuracy = accuracy_score(y_test_global, predictions)
print("\n--- Final Utterance-Level Results ---")
print(f"Accuracy (Weighted Average Recall): {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_global, predictions))

joblib.dump(svm_classifier, 'svm_emotion_model.pkl')
print("Complete model successfully trained and saved!")


Training Linear SVM on DTPM features...
Evaluating on unseen Test Set (Speakers 15 & 16)...

--- Final Utterance-Level Results ---
Accuracy (Weighted Average Recall): 69.29%

Classification Report:
              precision    recall  f1-score   support

           0       0.69      1.00      0.82        27
           1       0.81      0.57      0.67        23
           2       0.62      0.33      0.43        15
           3       0.46      0.35      0.40        17
           4       0.57      1.00      0.72        13
           5       0.81      0.81      0.81        16
           6       0.92      0.69      0.79        16

    accuracy                           0.69       127
   macro avg       0.70      0.68      0.66       127
weighted avg       0.71      0.69      0.67       127

Complete model successfully trained and saved!
